In [3]:
from pathlib import Path
import pandas as pd

photo_folder = Path("C:/Users/omistaja/Desktop/AS Project/Data/Photos/")

files = list(photo_folder.glob("*"))

records = []

for file in files:
    records.append({
        "filename": file.name,
        "path": str(file)
    })

import re
import pandas as pd

manifest = pd.DataFrame(records)

manifest.head(30)

,filename,path
0,S0002_360_001.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
1,S0002_360_002.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
2,S0002_360_003.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
3,S0002_360_004.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
4,S0002_360_005.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
5,S0002_360_006.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
6,S0002_360_007.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
7,S0002_360_008.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
8,S0002_360_009.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
9,S0002_360_010.jpg,C:\Users\omistaja\Desktop\AS Project\Data\Phot...


In [29]:
test_file = manifest.iloc[0]["path"]

print(test_file)

C:\Users\omistaja\Desktop\AS Project\Data\Photos\S0002_360_o\R0011073.JPG


In [30]:
from PIL import Image
from PIL.ExifTags import TAGS

with Image.open(test_file) as img:
    exif = img.getexif()

    print("Number of EXIF fields:", len(exif))

    for tag_id, value in exif.items():
        tag = TAGS.get(tag_id, tag_id)
        print(tag, ":", value)

Number of EXIF fields: 13
GPSInfo : 2230
ResolutionUnit : 2
ExifOffset : 344
ImageDescription :                                                                
Make : RICOH
Model : RICOH THETA Z1
Software : RICOH THETA Z1 Ver 3.50.2
Orientation : 1
DateTime : 2026:08:18 12:57:52
YCbCrPositioning : 1
Copyright :                         
XResolution : 300.0
YResolution : 300.0


manifest["capture_datetime_original"] = (
    manifest["path"].apply(get_timestamp)
)

manifest

manifest["capture_datetime_original"] = pd.to_datetime(
    manifest["capture_datetime_original"],
    format="%Y:%m:%d %H:%M:%S",
    errors="coerce"
)

manifest[
    ["filename", "capture_datetime_original"]
].head(10)

camera_offset_hours = 0

manifest["capture_datetime_corrected"] = (
    manifest["capture_datetime_original"]
    + pd.Timedelta(hours=camera_offset_hours)
)

manifest[
    [
        "filename",
        "capture_datetime_original",
        "capture_datetime_corrected"
    ]
].head(1500)

manifest.to_csv("S0002_photo_timestamps_corrected")

In [8]:
%pip install gpxpy

Note: you may need to restart the kernel to use updated packages.


from pathlib import Path
import gpxpy
import pandas as pd

gpx_path = Path(
    "C:/Users/omistaja/Desktop/AS Project/Data/Comaps/Wulai_District_08182026_S0002.gpx"
)

with open(gpx_path, "r", encoding="utf-8") as gpx_file:
    gpx = gpxpy.parse(gpx_file)

track_points = []

for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            track_points.append({
                "gpx_time": point.time,
                "latitude": point.latitude,
                "longitude": point.longitude,
                "elevation": point.elevation
            })

gpx_df = pd.DataFrame(track_points)

gpx_df["gpx_time_taiwan"] = (
    pd.to_datetime(gpx_df["gpx_time"], utc=True)
    .dt.tz_convert("Asia/Taipei")
)

gpx_df["gpx_time_local"] = (
    gpx_df["gpx_time_taiwan"]
    .dt.tz_localize(None)
)

gpx_df.head(600)

manifest[
    ["filename", "capture_datetime_corrected"]
].head(20)

gpx_df[
    ["gpx_time_taiwan", "latitude", "longitude"]
].head(445)

In [ ]:
manifest = manifest.sort_values(
    "capture_datetime_corrected"
).reset_index(drop=True)

gpx_df = gpx_df.sort_values(
    "gpx_time_local"
).reset_index(drop=True)

matched = pd.merge_asof(
    manifest,
    gpx_df[
        ["gpx_time_local", "latitude", "longitude"]
    ],
    left_on="capture_datetime_corrected",
    right_on="gpx_time_local",
    direction="nearest"
)

matched.head(445)

matched.to_csv("Photo coordinates2")

In [13]:
matched["time_difference_seconds"] = (
    matched["capture_datetime_corrected"]
    - matched["gpx_time_local"]
).abs().dt.total_seconds()

matched[
    [
        "filename",
        "capture_datetime_corrected",
        "gpx_time_local",
        "time_difference_seconds",
        "latitude",
        "longitude"
    ]
].head(20)

matched["time_difference_seconds"].describe()

matched["gpx_match_status"] = matched["time_difference_seconds"].apply(
    lambda x: "good" if x <= 30 else "review"
)

matched["gpx_match_status"].value_counts()

gpx_match_status
good      323
review    122
Name: count, dtype: int64

In [14]:
matched["time_difference_seconds"].describe()

pd.cut(
    matched["time_difference_seconds"],
    bins=[0, 5, 10, 30, 60, 120, 300, float("inf")],
    include_lowest=True
).value_counts().sort_index()

time_difference_seconds
(-0.001, 5.0]     109
(5.0, 10.0]        77
(10.0, 30.0]      137
(30.0, 60.0]       74
(60.0, 120.0]      41
(120.0, 300.0]      5
(300.0, inf]        2
Name: count, dtype: int64

In [15]:
matched[
    [
        "filename",
        "capture_datetime_corrected",
        "gpx_time_local",
        "time_difference_seconds",
        "latitude",
        "longitude"
    ]
].sort_values(
    "time_difference_seconds",
    ascending=False
).head(30)

,filename,capture_datetime_corrected,gpx_time_local,time_difference_seconds,latitude,longitude
0,IMG_0659.JPG,2026-08-18 12:56:10,2026-08-18 13:02:47,397.0,24.847607,121.55125
1,IMG_0660.JPG,2026-08-18 12:56:22,2026-08-18 13:02:47,385.0,24.847607,121.55125
2,IMG_0665.JPG,2026-08-18 12:58:50,2026-08-18 13:02:47,237.0,24.847607,121.55125
3,IMG_0666.JPG,2026-08-18 12:59:01,2026-08-18 13:02:47,226.0,24.847607,121.55125
4,IMG_0668.JPG,2026-08-18 13:00:03,2026-08-18 13:02:47,164.0,24.847607,121.55125
5,IMG_0669.JPG,2026-08-18 13:00:04,2026-08-18 13:02:47,163.0,24.847607,121.55125
134,IMG_0825.JPG,2026-08-18 13:39:51,2026-08-18 13:37:45,126.0,24.848721,121.55170
133,IMG_0824.JPG,2026-08-18 13:39:45,2026-08-18 13:37:45,120.0,24.848721,121.55170
135,IMG_0826.JPG,2026-08-18 13:41:09,2026-08-18 13:43:09,120.0,24.848801,121.55180
27,IMG_0706.JPG,2026-08-18 13:14:41,2026-08-18 13:12:46,115.0,24.847857,121.55138


In [123]:
target_time = pd.Timestamp("2026-08-18 07:58:50")

closest = gpx_df.iloc[
    (gpx_df["gpx_time_local"] - target_time)
    .abs()
    .argsort()[:1]
]

closest[
    ["gpx_time_local", "latitude", "longitude"]
]

,gpx_time_local,latitude,longitude
0,2026-08-18 13:02:47,24.847607,121.55125
